# PlastPrior: evidence, hazard, exposure and priority assessment

This notebook runs seven scientific stages, from frozen input inspection to final scores, eligible ranks and inclusive Top-N. Historical reproduction and audited analysis remain separate throughout. Fifteen model-identity exceptions retain their source records while their QSAR and Pov/LRTP evidence is withheld from analysis. Missing evidence is not filled with zero.

## Setup and reading guide

Follow the [installation instructions](README.md#installation) and [data preparation checklist](docs/data_preparation.md). The complete frozen input collection, including `ExposureReference/`, is required and is not supplied by a code clone. Public acquisition and distribution arrangements remain incomplete.

Select **PlastPrior (Python 3.12)**, use the project root as the working directory, and set `DATA_ROOT` and `REPORT_DIR` below. Choose an unused output directory, such as `Path("results") / "reproduction_nb_01"`, then restart the kernel and run all 17 code cells. Stage files in a reused directory can be replaced. The final export always uses a new directory.

The last local run and comparisons took about 13 minutes; stage files and the final package occupied about 9.2 GB before inputs and temporary files. These observations do not establish portable resource requirements. Native Windows and a fresh independent installation remain unverified.

Read [reproduction and verification](docs/reproduction.md) for execution and comparison steps, and [methods](docs/methods.md) for full mappings, thresholds and historical checks. Stage completion messages describe each stage boundary; all seven stages are implemented. Generated paths below are relative to `REPORT_DIR`. The final cell prints the CSV and SQLite paths for routine use.


In [ ]:
from pathlib import Path
from IPython.display import display

if not (Path.cwd() / "plastprior" / "inputs.py").is_file():
    raise RuntimeError("Run this notebook from the PlastPrior project directory.")

from plastprior.inputs import build_source_records, inspect_inputs
from plastprior.identity import resolve_identities
from plastprior.qsar import integrate_qsar
from plastprior.lrtp import integrate_lrtp
from plastprior.evidence import integrate_evidence
from plastprior.evidence_specialists import integrate_specialist_evidence
from plastprior.evidence_annotations import integrate_annotations
from plastprior.hazard import integrate_pm_hazard
from plastprior.hazard_bt import integrate_bt_hazard

DATA_ROOT = Path("data")
REPORT_DIR = Path("results")
from plastprior.exposure_inputs import integrate_exposure_evidence
import pandas as pd
from plastprior.exposure import integrate_exposure_scores
from plastprior.priority import integrate_priority
from plastprior.ranking import integrate_ranking


## Stage 1 — Input inspection

Inspect declared source files and selected worksheets. Versions and snapshot dates are recorded metadata; **Not recorded** identifies a provenance gap. Parsed rows are source records, not unique chemicals, and cannot be summed across overlapping inventories. Table roles distinguish candidate inventories, identity references and literature links.

Source-file and table checks must pass. Auxiliary inputs receive presence checks here; their content and fingerprints are checked by the stages that use them. The exposure reference bundle is also required for a complete run, even when the initial inspection passes.


In [ ]:
inspection = inspect_inputs(DATA_ROOT)
inspection.write_reports(REPORT_DIR)

display(inspection.sources[[
    "source", "filename", "version", "snapshot_date", "status"
]])
display(inspection.tables[[
    "source_id", "table_id", "table_role", "parsed_rows", "nonempty_rows", "columns", "status"
]])

**Review the inspection outcome.** Source errors stop execution after reports have been saved. Missing auxiliary inputs remain visible but do not invalidate this Notebook's source-only inspection. An inspection pass does not establish scientific validity or full-run readiness.

`source_inventory.csv`, `table_dimensions.csv` and `supporting_inputs.csv` retain recorded references, observed fingerprints, table declarations and diagnostics under `REPORT_DIR`. A hash identifies file bytes; it does not independently certify a source version.


In [ ]:
display(inspection.supporting[["input_group", "status"]])
if not inspection.supporting_present:
    print("Supporting inputs are incomplete; see supporting_inputs.csv.")

for label, frame in [("Source files", inspection.sources), ("Source tables", inspection.tables)]:
    failures = frame.loc[frame["status"].ne("OK")]
    if not failures.empty:
        print(label)
        display(failures)

reader_notes = inspection.tables.loc[
    inspection.tables["reader_warnings"].ne(""), ["table_id", "reader_warnings"]
]
if not reader_notes.empty:
    print("Workbook-reader notes")
    display(reader_notes)

inspection.require_sources()
print("Stage 1 source inspection passed. Auxiliary content validation remains pending.")

## Stage 2 — Source assembly and chemical identity

Assemble 11 inventory tables using [explicit field mappings](plastprior/sources.json). PubChemLite, NORMAN SusDat and the LitChemPlast literature-link table are outside the candidate inventory. PlastChem uses isomeric SMILES when supplied, with canonical SMILES as fallback. The historical MPLEACH inclusion rule requires reported InChIKey and SMILES; excluded rows retain reasons.

The tables below count included source records and identifier availability, not unique or confirmed chemicals. Original mapped fields, placeholders, native identifiers and unselected fallback values remain traceable. Outputs under `stage2/` preserve record-level provenance before identity matching.


In [ ]:
source_records = build_source_records(DATA_ROOT)
source_records.write_reports(REPORT_DIR / "stage2")

display(source_records.summary[[
    "source_id", "table_id", "parsed_records", "included_records", "excluded_records"
]])
display(source_records.summary[[
    "table_id", "included_records", "name_available", "cas_available",
    "smiles_available", "inchikey_available"
]])
excluded = source_records.records.loc[~source_records.records["included"]]
if excluded.empty:
    print("No inventory records were excluded under the declared inclusion rules.")
else:
    display(excluded.groupby(["table_id", "exclusion_reason"]).size().rename("records").reset_index())
print("Source record assembly complete; proceed to identity resolution below.")

**Resolve structures and retain source links.** The migration policy uses acceptable reported SMILES first, then full-InChIKey lookup in PubChemLite, PlastChem and NORMAN, then CAS lookup when eligible candidates agree on one full key. Tiers A–D indicate route precedence, not identification confidence.

Aggregation uses the full structure-derived InChIKey. Every member record is retained; records without valid keys cannot enter the compound inventory. The non-isomeric QSAR key is an annotation and does not merge stereoisomers. Name matching, desalting and tautomer normalisation are outside this identity policy.

Review resolution routes, unresolved records and reference conflicts below. Selection decisions, reference rows and compound memberships remain in `stage2/`. [Historical regression](docs/methods.md#historical-regression-validation) requires separate frozen baseline files; successful execution alone does not establish historical agreement.


In [ ]:
identities = resolve_identities(source_records, DATA_ROOT)
identities.write_reports(REPORT_DIR / "stage2")

for measure, count in identities.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(identities.summary)
display(identities.reference_summary)
print("Stage 2 complete. Review unresolved records and audit reports before proceeding to QSAR integration.")


## Stage 3A — Frozen QSAR evidence

Apply the historical structure-screening policy without changing Stage 2 identities. Full identities can share a non-isomeric QSAR representation without being merged. The regenerated input batch and all 20 report fingerprints must match before row-based associations are imported. VEGA is not rerun.

[Model declarations](plastprior/qsar_models.json) specify versions, fields, units and quality measures. Missing ADI is not zero, and low ADI remains available for later eligibility assessment. Raw cells from variable-width reports remain in the layout audit.

The table below identifies report/input structure discrepancies. The 15 exceptions remain `review_required`, with historical predictions retained for audit and source/report links preserved under `stage3/`. Formula and radical differences describe observations; their origin and chemical correctness remain unresolved. Prediction availability is not evidence of identity confirmation or model reliability.


In [ ]:
qsar = integrate_qsar(identities, DATA_ROOT)
qsar.write_reports(REPORT_DIR / "stage3")

for measure, count in qsar.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(qsar.model_summary[[
    "model_name", "model_metadata", "rows", "identity_review_rows", "error_assessment_rows", "adi_provided"
]])
exceptions = qsar.identity_audit.loc[~qsar.identity_audit["identity_verified"]]
display(exceptions[[
    "vega_row_no", "input_formula", "report_formula", "input_radical_electrons",
    "report_radical_electrons", "reason", "identity_status"
]])
print("Stage 3A complete. Identity exceptions remain pending scientific review; Pov/LRTP has not been calculated.")


## Stage 3B — Pov/LRTP and identity isolation

Reproduce the archived three-compartment calculation from six declared QSAR properties. Unit conversions, environmental constants, reference lines and operation order follow the [stage methods](docs/methods.md). The calculation has not been independently validated against the official OECD workbook.

Historical results retain emission scenarios and original values. Analysis withholds five converted inputs, four numerical outputs and three threshold flags for unverified identities. Missing values remain distinct from zero or False; identity reasons and source/report links persist under `stage3/lrtp/`.

The table displays excluded identities and historical values for audit only. Numerical availability and identity checks do not establish adequate QSAR reliability; ADI and source warnings remain relevant to later assessment.


In [ ]:
lrtp = integrate_lrtp(REPORT_DIR / "stage3")
lrtp.write_reports(REPORT_DIR / "stage3" / "lrtp")

for measure, count in lrtp.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(lrtp.compounds.groupby(
    ["calculation_status", "qsar_identity_status", "lrtp_available"], dropna=False
).size().rename("compounds").reset_index())
excluded = lrtp.compounds.loc[
    lrtp.compounds["lrtp_exclusion_reason"].eq("identity_review_required"),
    ["inchikey", "vega_row_no", "qsar_identity_reason", "calculation_status", "lrtp_exclusion_reason"]
].merge(lrtp.historical[["inchikey", "LRTP_POV_days", "LRTP_CTD_km", "LRTP_TE_percent"]],
        on="inchikey", validate="one_to_one")
display(excluded.rename(columns={
    "LRTP_POV_days": "Historical Pov (days; audit only)",
    "LRTP_CTD_km": "Historical CTD (km; audit only)",
    "LRTP_TE_percent": "Historical TE (%; audit only)"
}))
print("Stage 3B complete. Identity-review results are withheld from analysis; historical values remain available for audit.")


## Stage 4A — Basic evidence and field mapping

Assemble basic source evidence using declared fields, source selection rules and identity gates. Retain attempted associations, selected rows, original values and exclusion reasons. Independent database observations remain separate from model predictions.

For the 15 model-identity exceptions, 57 QSAR fields and 12 Pov/LRTP fields remain unavailable in analysis. Their historical values and source records remain traceable. The tables below show association decisions, mapping coverage and isolation actions; detailed records are saved under `stage4/`.


In [ ]:
evidence = integrate_evidence(REPORT_DIR / "stage2", REPORT_DIR / "stage3" / "lrtp", DATA_ROOT)
evidence.write_reports(REPORT_DIR / "stage4")

for measure, count in evidence.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
import pandas as pd
attempts = pd.DataFrame(evidence.attempts)
display(attempts.groupby(["db", "identity_gate_status"]).size().unstack(fill_value=0))
display(evidence.field_mapping.groupby("source").agg(
    declared_field_mappings=("source_column", "size"),
    available_field_mappings=("column_present", "sum")
))
display(evidence.isolation.groupby(["exclusion_reason", "action"]).size().rename("field_entries").reset_index())
print("Stage 4A complete. Identity-review predictions remain isolated; specialist enrichment and scoring are deferred.")


## Stage 4B — Literature associations and human detection

LitChemPlast uses exact full InChIKey before guarded CAS fallback for rows without a valid full key. Counts distinguish unique chemical relationships (`LitChem_id`) from publications (`Lit_id`); declared source counts are audited separately.

FCChumon requires an unambiguous target CAS and exactly one sheet/CAS candidate. S1 database entries and S2 studies use different detection definitions. Historical within-sheet maxima are retained; ambiguous cross-sheet associations are excluded. A selected source with no positive count differs from unavailable evidence, whose analysis detection flag remains missing.

Independent eligible literature and detection evidence remains usable for the 15 model-identity exceptions. No identity is repaired here. Review matching methods, availability and the exception subset below; source rows, CAS links and selection/exclusion reasons are retained under `stage4/4b/`.


In [ ]:
specialists = integrate_specialist_evidence(REPORT_DIR / "stage2", REPORT_DIR / "stage4", DATA_ROOT)
specialists.write_reports(REPORT_DIR / "stage4" / "4b")

for measure, count in specialists.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(pd.Series(specialists.manifest["litchemplast_methods"], name="compounds").rename_axis("LitChemPlast match method").to_frame())
display(specialists.compounds.groupby("fcchumon_evidence_status").size().rename("compounds").to_frame())
display(specialists.anomalies[[
    "vega_row_no", "identity_reason", "litchemplast_match_method",
    "litchemplast_linked_record_count", "litchemplast_unique_publication_count",
    "fcchumon_evidence_status", "prediction_exclusion_reason"
]])
print("Stage 4B complete. Independent source evidence is retained; identity-review predictions remain unavailable.")


## Stage 4C — Analytical predictions and regulatory-list associations

NORMAN SusDat analytical-platform predictions use the canonical full InChIKey and declared candidate precedence. They describe predicted analytical amenability, not observed detection, and do not enter scoring.

GIC follows full-key lookup when available, CID bridges and then resolved names. This frozen inventory has no queryable InChIKeys. Name fallback, multiple identifiers and tag expansion through shared identifiers remain explicit review flags. An association does not establish chemical identity or current regulatory applicability.

Inventory and bridge inputs are fingerprinted. The metadata CSV's historical byte hash and historical GIC package version were not recorded; current bytes and a repository revision are pinned. Effective metadata fields can be compared with the frozen audit, but this does not recover missing provenance. Source rows, attempted paths and selection/exclusion reasons remain under `stage4/4c/`. All 15 model-identity exceptions retain their QSAR/Pov-LRTP exclusions.


In [ ]:
annotations = integrate_annotations(REPORT_DIR / "stage4" / "4b", DATA_ROOT)
annotations.write_reports(REPORT_DIR / "stage4" / "4c")

for measure, count in annotations.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(annotations.compounds.groupby("norman_prediction_available").size().rename("compounds").to_frame())
display(annotations.compounds.groupby(
    ["gic_match_method", "gic_name_review_required", "gic_multiple_cids", "gic_multiple_sids", "gic_shared_identifier_tag_expansion"]
).size().rename("compounds").reset_index())
display(annotations.anomalies[[
    "inchikey", "vega_row_no", "qsar_identity_reason", "qsar_evidence_available", "lrtp_available",
    "norman_prediction_available", "gic_match_method", "gic_name_review_required", "gic_shared_identifier_tag_expansion"
]])
print("Metadata historical byte hash: not recorded; frozen effective-field validation is required.")
print("Stage 4C complete. Identity reviews remain pending; no downstream scoring performed.")


## Stage 5A — Persistence and mobility

P uses the maximum usable IRFMN compartment assessment (ADI ≥ 0.5). M uses the maximum usable OPERA aquatic and OECD LRTP pathway score. Usable independent evidence, including zero, precedes curated fallback. Missing evidence remains unavailable. [Hazard policy](plastprior/hazard_policy.json) defines mappings, units and cutoffs.

Historical reproduction restores frozen model inputs solely for comparison. Analysis continues to exclude QSAR/Pov-LRTP evidence for the 15 identity-review compounds. Independent curated P/M evidence could remain eligible, but none is available for this cohort. Their 58 source links and exclusion reasons remain traceable.

Review historical versus analysis availability below. Detailed evidence decisions and field changes are saved under `stage5/`. GIC review flags and NORMAN platform predictions do not enter P/M scoring.


In [ ]:
hazard = integrate_pm_hazard(REPORT_DIR / "stage4" / "4c")
hazard.write_reports(REPORT_DIR / "stage5")

display(hazard.summary)
review_keys = hazard.compounds.loc[
    hazard.compounds["qsar_identity_status"].eq("review_required"), "inchikey"
]
historical_review = hazard.historical.loc[
    hazard.historical["inchikey"].isin(review_keys),
    ["inchikey", "Persistence_score", "Mobility_score"]
].rename(columns={"Persistence_score": "historical_P", "Mobility_score": "historical_M"})
display(historical_review.merge(hazard.compounds[[
    "inchikey", "Persistence_score", "Mobility_score",
    "persistence_unavailable_reason", "mobility_unavailable_reason"
]], on="inchikey", validate="one_to_one"))
display(pd.Series({
    "GIC name matches requiring review": hazard.manifest["counts"]["gic_name_review_required"],
    "GIC shared-identifier tag expansions requiring review": hazard.manifest["counts"]["gic_tag_expansion_review_required"],
    "Identity-review source links retained": hazard.manifest["counts"]["identity_review_source_links"]
}, name="count").to_frame())
print("Stage 5A complete. GIC and NORMAN annotations do not enter P/M scoring.")


## Stage 5B — Bioaccumulation and toxicity

B selects the first usable BCF model in Arnot–Gobas, CAESAR, Meylan order (ADI ≥ 0.5), then takes the maximum of that score and the inherited Cl/Br structural rule. T uses the maximum usable endpoint score. Ames has no ADI field in the frozen configuration but still requires eligible identity. Usable independent evidence, including zero, precedes curated fallback; unavailable evidence remains missing.

The structural pathway uses locally generated `qsar_smiles`, not the model-report `Basic | Smile_std`. Source structures, representative record IDs, standardisation changes and RDKit version are retained. The 15 model-identity exceptions keep independent structure/database evidence: one has a structural B score and five have curated T fallback. Three T scores increase after historical QSAR zeros are excluded; source changes are shown below.

GIC flags and NORMAN annotations do not enter B/T scoring. Historical and analysis evidence, mappings and changes remain under `stage5/5b/`. These inherited screening rules are not independently validated regulatory classifications.


In [ ]:
hazard_bt = integrate_bt_hazard(REPORT_DIR / "stage5")
hazard_bt.write_reports(REPORT_DIR / "stage5" / "5b")

display(hazard_bt.summary)
review_changes = hazard_bt.impacts.loc[
    hazard_bt.impacts["field"].isin(["Bioaccumulation_score", "Toxicity_score"]),
    ["inchikey", "field", "historical_value", "analysis_value", "historical_basis", "analysis_basis", "change_reason"]
]
display(review_changes)
display(hazard_bt.structures.loc[
    hazard_bt.structures["text_changed_by_local_screening"],
    ["inchikey", "source_smiles_standard", "scoring_smiles", "structural_score"]
])
print("GIC review flags retained:", hazard_bt.manifest["counts"]["gic_name_review_required"],
      "name matches and", hazard_bt.manifest["counts"]["gic_tag_expansion_review_required"], "shared-identifier expansion.")
print("Stage 5B complete. P, B, M and T remain separate dimensions; no aggregate score is calculated.")


## Stage 6A — Exposure evidence and eligibility

Import fingerprinted production/market tables and check their recorded raw-file hashes. These are frozen derived inputs; upstream transformations are not rerun. Observations retain native fields, source identifiers and exclusion reasons. Qualitative market evidence requires an accepted identity mapping independently of numerical availability.

GIC name-only matches remain under review. Inventory eligibility requires direct local tag support on at least one matched SID; the historical metadata provenance gap persists. Negative quantities and failed or out-of-scope patent/PubMed queries retain explicit audit status rather than becoming valid zeros.

The 15 model-identity exceptions retain their model exclusions; independent exposure evidence is evaluated separately. Review source coverage and eligibility below. Detailed evidence is saved under `stage6/6a/`; this step audits inputs without calculating exposure scores.


In [ ]:
exposure = integrate_exposure_evidence(REPORT_DIR / "stage5" / "5b", DATA_ROOT)
exposure.write_reports(REPORT_DIR / "stage6" / "6a")

display(exposure.summary)
display(pd.read_csv(REPORT_DIR / "stage6" / "6a" / "market_source_summary.csv"))
print("Exposure evidence audited; no exposure scores calculated.")


## Stage 6B — Exposure dimension scores

Assess five dimensions: production/market scale, patent activity, PubMed query frequency, food-contact/human detection, and industrial-inventory association. Food contact and human detection are separate binary contributions within one dimension. Historical selection and zero-filling rules remain in the reproduction view only.

Analysis applies Stage 6A eligibility. Ambiguous market identities, unresolved GIC name associations, invalid quantities and failed queries cannot supply the affected evidence. FCChumon evidence with no valid query CAS or a rejected ambiguous-CAS association does not establish a zero detection indicator.

A formal total requires all five dimensions. Missing components remain missing; the observed-component sum and coverage are retained without renormalisation. Zero denotes the defined indicator, not absence of real-world exposure. Independent eligible evidence remains usable for model-review compounds. Scores, source decisions and changes are saved under `stage6/6b/`.


In [ ]:
exposure_scores = integrate_exposure_scores(REPORT_DIR / "stage6" / "6a", DATA_ROOT)
exposure_scores.write_reports(REPORT_DIR / "stage6" / "6b")

display(exposure_scores.summary)
print(exposure_scores.manifest["counts"])
print("Review missing dimensions and evidence changes before interpreting exposure totals.")


## Stage 7A — Aggregate scores, completeness and tiers

Use the frozen weighted hazard sum (2P + M + 2B + 3T), additive exposure score and dual-gate priority tiers. Historical zero filling is restricted to the reproduction view. Analysis requires all four hazard and five exposure dimensions; missing totals and the `Insufficient evidence` tier preserve uncertainty. Observed contributions are not renormalised.

Model-identity quarantine and the upstream decisions on GIC associations, negative quantities and failed queries remain in force. Independent eligible evidence retains its source links. Complete-case results cannot be assumed representative of all compounds; low priority does not establish safety.

Review completeness, tier counts and identity-review records below. Detailed decisions and changes are saved under `stage7/7a/`. Ranking is performed in the next step.


In [ ]:
priority = integrate_priority(REPORT_DIR / "stage6" / "6b")
priority.write_reports(REPORT_DIR / "stage7" / "7a")

display(priority.summary)
display(pd.DataFrame([priority.manifest["counts"]]))
display(priority.compounds.loc[
    priority.compounds["qsar_identity_status"].eq("review_required"),
    ["inchikey", "Hazard_available_dimensions", "Exposure_available_dimensions",
     "Priority_eligible", "Priority_tier", "Priority_ineligibility_reasons"]
])


## Stage 7B — Eligible ranking and inclusive Top-N

Only complete Stage 7A assessments enter ranking. Equal unrounded scores share competition rank (1, 2, 2, 4). Full InChIKey supplies stable display order within ties, not a different scientific rank. Top-N includes the entire boundary tie group, so its size can exceed N. Tiers retain their hazard/exposure gates and do not determine sorting.

Historical Top-100 reproduction remains separate. Unranked records retain null ranks and null Top-N selection, with identity/GIC flags and missing reasons. Rank changes distinguish eligibility-pool changes from score changes; a better rank does not itself establish greater measured risk.

The preview below is shorter than the scientific Top-N export. Complete ranking, inclusive Top-N and unranked review tables are saved under `stage7/7b/` and included in the final package.


In [ ]:
rankings = integrate_ranking(REPORT_DIR / "stage7" / "7a", top_n=100)
rankings.write_reports(REPORT_DIR / "stage7" / "7b")

display(pd.DataFrame([rankings.summary]))
display(rankings.tables["top_priority_including_ties"][[
    "Priority_rank", "Priority_tie_size", "inchikey", "name_resolved",
    "Hazard_score", "Exposure_score", "Priority_score", "Priority_tier"
]].head(10))
display(rankings.tables["unranked_review"][[
    "inchikey", "Priority_rank", "Priority_ineligibility_reasons"
]].head(5))


## Final results

The last cell exports a publication package into a new `REPORT_DIR / "final" / <run_id>` directory and prints their paths. **Start with the CSV** for a 58-field publication table with a data dictionary and README; use **SQLite** for all 251 final fields, historical results, source links and necessary audit information. The verified frozen collection contains 15,125 compounds, 4,244 eligible ranks, 107 inclusive Top-100 members and 10,881 unranked records.

SQLite shares repeated audit bodies and stores intermediate values as differences from final values. Explicitly out-of-universe Stage 6A patent-reference bodies are omitted with coverage counts and row ranges retained. Original source bytes and whole-file stage backups are external. See the [storage contract and decoding helpers](docs/methods.md#final-results) for reconstruction limits.

Use `ORDER BY Priority_rank, inchikey` for ranked queries; `top_priority` includes boundary ties and `unranked_review` retains incomplete assessments. Missing numeric values remain NULL. Export verification checks the package against the current stage outputs; it does not establish independent scientific validity or agreement with a separate historical baseline.

[Verification guidance](docs/reproduction.md#verification) explains comparison criteria. [Retained local results](docs/results_layout.md) are available only where those ignored files are present; public result distribution remains unresolved. Export does not remove existing databases or intermediate files.


In [ ]:
from plastprior.export import export_results

final_package = export_results(REPORT_DIR)
display(pd.DataFrame([{key: final_package[key] for key in [
    "status", "compounds", "eligible_compounds", "top_n_compounds", "unranked_compounds"
]}]))
print("Final CSV:", final_package["csv"])
print("Final SQLite:", final_package["sqlite"])
print("Dashboard SQLite:", final_package["dashboard_sqlite"])
print("Data dictionary:", final_package["dictionary"])
